In [ ]:
#checking precip yay!!!
import numpy as np 
from matplotlib import pyplot as plt
import random as rand
from scipy.optimize import fsolve
import scipy
import math
import matplotlib
from scipy.optimize import root
from scipy.special import lambertw
from scipy.interpolate import interp1d
from scipy.integrate import quad
from scipy import integrate
from importlib import reload
from scipy.integrate import cumulative_trapezoid 

#importing py files
import ParamsECO1
import Functionsdiurnal1
import Atmosphericdynamics1
reload(Atmosphericdynamics1)
reload(Functionsdiurnal1)
reload(ParamsECO1)
from ParamsECO1 import *
from Functionsdiurnal1 import *
from Atmosphericdynamics1 import *

In [ ]:
#precip and cape graph (Figure 5)

import numpy as np
import matplotlib.pyplot as plt


#configuration
epsilon = 0.1 #precip eff. 
Pa      = 101325         # surface pressure [Pa]
Rv, Rd, Tref = 461, 287, 273

# grid of abl-near surface values 
T0s = np.linspace(290, 305, 200)          # K
q0s = np.linspace(0.010, 0.025, 200)      # kg/kg
TT, QQ = np.meshgrid(T0s, q0s)


#full vector
CAPEg = np.full(TT.shape, np.nan)
PRECg = np.full(TT.shape, np.nan)

# Tetens qsat -> mask supersaturated surface air (the blank upper-left region), not possible 
def qsat(T, P):
    es = 611.2 * np.exp(17.67 * (T - 273.15) / (T - 29.65))   # Pa
    return (Rd / Rv) * es / (P - (1 - Rd / Rv) * es)


#run through
for i in range(TT.shape[0]):
    for j in range(TT.shape[1]):
        T0, q0 = TT[i, j], QQ[i, j]
        if q0 >= qsat(T0, Pa):            # surface already saturated -> skip
            continue
        try:
            zl = zlcl(T0, q0)
            T_tp, P_tp = tp(zl, T0)
            PRECIP, PGROUND, Tmp1, Tfv1, CAPE = RAINC(
                epsilon, T_tp, P_tp, zl,
                17000, -2.1e-6, 0.011, 10,
                T0, q0, Rv=Rv, Rd=Rd, Tref=Tref, Pa=Pa)
            if np.isfinite(CAPE) and CAPE > 0:
                CAPEg[i, j] = CAPE
                PRECg[i, j] = PRECIP*1200 + PGROUND
        except Exception:
            pass

In [ ]:

plt.rcParams.update({
    "font.family": "sans",
    "mathtext.fontset": "dejavusans",
    "font.size": 15,
    "axes.labelsize": 17,
    "legend.fontsize": 13,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "axes.linewidth": 1.2,
})

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 5),dpi=300)

def panel(ax, Z, levels, label, cbar_ticks):
    cs = ax.contour(
        TT, QQ, Z,
        levels=levels,
        cmap='viridis',
        linewidths=3
    )

    # explicit saturation envelope (top boundary only)
    Tline = np.linspace(290, 305, 300)
    qline = qsat(Tline, Pa)
    m = (qline >= 0.010) & (qline <= 0.025)
    ax.plot(Tline[m], qline[m], 'k', linewidth=3.5)

    # colorbar with clean, evenly spaced integer ticks
    cb = fig.colorbar(cs, ax=ax, ticks=cbar_ticks)
    cb.set_label(label)

    # force whole-number labels
    cb.ax.set_yticklabels([f'{int(x)}' for x in cbar_ticks])

    ax.set_xlabel('Temperature (K)')
    ax.set_ylabel('Specific Humidity q (kg kg$^{-1}$)')
    ax.set_xlim(290, 305)
    ax.set_ylim(0.010, 0.025)


# CAPE: contours every 1000 J/kg, labels every 2000
cape_levels = np.arange(0, 16001, 800)
cape_ticks  = np.arange(0, 16001, 2000)

# Precipitation: contours every 10 mm, labels every 20 mm
prec_levels = np.arange(0, 141, 5)
prec_ticks  = np.arange(0, 141, 20)

panel(
    ax1, CAPEg,
    cape_levels,
    'CAPE (J kg$^{-1}$)',
    cape_ticks
)

panel(
    ax2, PRECg,
    prec_levels,
    'Precipitation (mm)',
    prec_ticks
)

plt.tight_layout()
plt.savefig("CAPEPRECIPnew")